# Fair-Explainable Clustering: from Fairness to Explainability
**Thesis Pipeline — Harish Sharma**

---
### Pipeline Overview
1. Configuration & Imports
2. Data Loading & Preprocessing
3. **Twagner Quadtree Builder** (replaces simple greedy fairlets)
4. **Twagner Tree Fairlet Decomposition** (distance-aware, no leftover drops)
5. Fairlet Medoid Computation
6. K-Medoids (PAM)
7. K-Medians
8. Label Assignment from Fairlets
9. Fairness Metrics
10. Explainability Metrics
11. Unified Evaluation Function
12. **Baseline: K-Means** → Metrics
13. **Baseline: K-Medians** → Metrics
14. **Twagner Fairlets + K-Medians** → Metrics
15. Explainability Analysis (Decision Rules)
16. Summary Comparison Table

---
## Cell 1 — Configuration & Imports

In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist

# ============================================================
# CONFIG
# ============================================================
K_CLUSTERS   = 5
P            = 1       # balance parameter (minority ratio numerator)
Q            = 2       # balance parameter (minority ratio denominator)
RANDOM_STATE = 42
SAMPLE_SIZE  = 30000   # set to None to use full dataset
EPSILON      = 0.0001  # quadtree cell-too-small threshold (from Twagner)

DATA_PATH = r'D:\Thesis\Fair_explainable_cluster_updated_2nd_feb\data\bank-full.csv'

print("Imports and config loaded successfully.")

Imports and config loaded successfully.


---
## Cell 2 — Data Loading & Preprocessing

In [2]:
def load_data():
    """
    Load the Bank Marketing dataset.
    Sensitive attribute: married=1, others=0.
    Returns:
        X_dense       : dense numpy float64 array (required by Twagner quadtree)
        sensitive     : binary numpy array
        feature_names : list of strings
    """
    df = pd.read_csv(DATA_PATH)

    # Sensitive attribute: married = 1, others = 0
    # df['sensitive'] = df['marital'].apply(lambda x: 1 if x == 'married' else 0)
    df['sensitive'] = df['housing'].apply(lambda x: 1 if x == 'yes' else 0)



    drop_cols = ['housing', 'sensitive', 'y']
    X_raw     = df.drop(columns=drop_cols)
    sensitive = df['sensitive'].values

    # Optional sampling for speed
    if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(X_raw):
        np.random.seed(RANDOM_STATE)
        idx       = np.random.choice(len(X_raw), SAMPLE_SIZE, replace=False)
        X_raw     = X_raw.iloc[idx].reset_index(drop=True)
        sensitive = sensitive[idx]

    # Separate numeric and categorical columns
    categorical_cols = X_raw.select_dtypes(include=['object']).columns
    numeric_cols     = X_raw.select_dtypes(include=['number']).columns

    preprocessor = ColumnTransformer([
        ('num', StandardScaler(),                      numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

    X_processed   = preprocessor.fit_transform(X_raw)
    feature_names = list(preprocessor.get_feature_names_out())

    # Twagner's quadtree requires a dense float64 numpy array
    if hasattr(X_processed, 'toarray'):
        X_dense = X_processed.toarray().astype(np.float64)
    else:
        X_dense = np.array(X_processed, dtype=np.float64)

    return X_dense, sensitive, feature_names

In [3]:
# Run data loading
X, sensitive, feature_names = load_data()

print(f"Dataset shape      : {X.shape}")
print(f"Sensitive (married): {sensitive.sum()} / {len(sensitive)} "
      f"({sensitive.mean()*100:.1f}%)")
print(f"Number of features : {len(feature_names)}")
print(f"Array type         : {type(X).__name__}, dtype={X.dtype}")

Dataset shape      : (30000, 48)
Sensitive (married): 16683 / 30000 (55.6%)
Number of features : 48
Array type         : ndarray, dtype=float64


C:\Users\91773\AppData\Local\Temp\ipykernel_7872\1630972817.py:30: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_raw.select_dtypes(include=['object']).columns


---
## Cell 3 — Twagner: TreeNode & Quadtree Builder

**Why this replaces the simple greedy approach:**
- Simple greedy paired points randomly with no regard for distance
- Twagner's quadtree recursively partitions the feature space into hypercubes
- Points that are **geometrically close** end up in the same tree node
- Fairlets are formed **within nearby nodes first**, minimising intra-fairlet cost
- Leftover points are **bubbled up** to parent nodes — nothing is dropped

In [4]:
# ── Twagner TreeNode ─────────────────────────────────────────
class TreeNode:
    """
    Node in the quadtree used for fairlet decomposition.
    Each node stores the data indices in its hypercube region
    and auxiliary red/blue lists populated bottom-up.
    (Directly from Twagner / Chierichetti et al. NIPS 2017)
    """

    def __init__(self):
        self.children = []

    def set_cluster(self, cluster):
        self.cluster = cluster

    def add_child(self, child):
        self.children.append(child)

    def populate_colors(self, colors):
        """
        Populate auxiliary red/blue index lists for each node, bottom-up.
        'colors' is a list where 0 = blue (non-sensitive), 1 = red (sensitive).
        NOTE: Twagner uses 0=red, 1=blue — we keep his convention internally
              and map our sensitive array (1=married) accordingly at call time.
        """
        self.reds  = []
        self.blues = []
        if len(self.children) == 0:
            # Leaf node: assign points by color
            for i in self.cluster:
                if colors[i] == 0:
                    self.reds.append(i)
                else:
                    self.blues.append(i)
        else:
            # Internal node: aggregate children
            for child in self.children:
                child.populate_colors(colors)
                self.reds.extend(child.reds)
                self.blues.extend(child.blues)

In [5]:
# ── Twagner Quadtree Builder ─────────────────────────────────
def build_quadtree(dataset, max_levels=0, random_shift=True):
    """
    Build a quadtree (k-d tree style) over the dataset.
    Recursively splits the bounding hypercube at midpoints.

    Parameters
    ----------
    dataset      : (n, d) float64 numpy array
    max_levels   : max depth (0 = no limit, splits until singletons)
    random_shift : apply a random shift to reduce worst-case behaviour
                   (standard technique from Chierichetti et al.)

    Returns
    -------
    root TreeNode
    """
    dimension = dataset.shape[1]
    lower = np.amin(dataset, axis=0)
    upper = np.amax(dataset, axis=0)

    shift = np.zeros(dimension)
    if random_shift:
        np.random.seed(RANDOM_STATE)
        for d in range(dimension):
            spread    = upper[d] - lower[d]
            shift[d]  = np.random.uniform(0, spread)
            upper[d] += spread

    return _build_quadtree_aux(
        dataset, list(range(dataset.shape[0])),
        lower, upper, max_levels, shift
    )


def _build_quadtree_aux(dataset, cluster, lower, upper, max_levels, shift):
    """
    Recursive helper — splits current hypercube at midpoint in all dimensions.
    Stops when: max_levels==1, cluster is singleton, or cell is too small.
    """
    dimension = dataset.shape[1]

    # Check if cell is too small to split further
    cell_too_small = all(upper[i] - lower[i] <= EPSILON for i in range(dimension))

    node = TreeNode()
    if max_levels == 1 or len(cluster) <= 1 or cell_too_small:
        # Leaf node
        node.set_cluster(cluster)
        return node

    # Split at midpoint in all dimensions simultaneously
    midpoint    = 0.5 * (lower + upper)
    subclusters = defaultdict(list)
    for i in cluster:
        # Each point goes into the sub-cell based on which side of midpoint it falls
        key = tuple(dataset[i, d] + shift[d] <= midpoint[d] for d in range(dimension))
        subclusters[key].append(i)

    for edge, subcluster in subclusters.items():
        sub_lower = np.zeros(dimension)
        sub_upper = np.zeros(dimension)
        for d in range(dimension):
            if edge[d]:
                sub_lower[d] = lower[d]
                sub_upper[d] = midpoint[d]
            else:
                sub_lower[d] = midpoint[d]
                sub_upper[d] = upper[d]
        node.add_child(
            _build_quadtree_aux(dataset, subcluster, sub_lower, sub_upper,
                                max_levels - 1, shift)
        )
    return node


print("Quadtree builder defined.")

Quadtree builder defined.


---
## Cell 4 — Twagner: Tree Fairlet Decomposition

**How leftover points are handled (vs simple greedy):**
- Simple greedy: leftover unmatched points are either dropped or form imbalanced singletons
- Twagner: uses 3-phase node processing to **bubble excess points UP** to parent nodes
  - Phase 1 — must-remove: points that would make child too imbalanced are lifted
  - Phase 2 — may-remove: optional points lifted to help the parent become balanced
  - Phase 3 — unsaturated: excess from incomplete fairlets lifted
- Parent collects all lifted points and forms its own balanced fairlets from them
- **No point is ever dropped or left unassigned**

In [6]:
# ── Twagner Fairlet Helper Functions ─────────────────────────

def _balanced(p, q, r, b):
    """
    Check if (r, b) satisfies the (p, q) balance condition.
    Balance condition: min(r/b, b/r) >= p/q
    """
    if r == 0 and b == 0:
        return True
    if r == 0 or b == 0:
        return False
    return min(r * 1. / b, b * 1. / r) >= p * 1. / q


def _make_fairlet(points, dataset, FAIRLETS, FAIRLET_CENTERS):
    """
    Add a fairlet to the decomposition.
    Finds the medoid (minimises total distance to all other points in fairlet)
    and records it as the fairlet center.
    Returns the intra-fairlet cost (sum of distances to medoid).
    """
    FAIRLETS.append(points)
    # cost_list[i] = total distance from points[i] to all other points
    cost_list = [
        sum(np.linalg.norm(dataset[center, :] - dataset[point, :]) for point in points)
        for center in points
    ]
    cost, center_pos = min((c, i) for (i, c) in enumerate(cost_list))
    FAIRLET_CENTERS.append(points[center_pos])
    return cost


def _basic_fairlet_decomposition(p, q, blues, reds, dataset, FAIRLETS, FAIRLET_CENTERS):
    """
    Vanilla (p,q)-fairlet decomposition of a balanced set of points.
    Implements Lemma 3 from Chierichetti et al. NIPS 2017 (via Twagner).

    Assumes: len(reds) >= len(blues), and the set is already balanced.
    Handles three cases for leftovers so nothing is dropped:
      1. Total remaining fits in one fairlet → merge all
      2. Imbalanced remainder with enough blues → form one mixed fairlet
      3. Equal remainder → pair 1:1

    Returns total intra-fairlet cost.
    """
    assert p <= q, "Balance parameters must satisfy p <= q"

    # Ensure reds is the larger group (Twagner convention)
    if len(reds) < len(blues):
        reds, blues = blues, reds

    R = len(reds)
    B = len(blues)

    assert _balanced(p, q, R, B), \
        f"Input sets are unbalanced: reds={R}, blues={B}"

    if R == 0 and B == 0:
        return 0

    r0   = 0
    b0   = 0
    cost = 0

    # Main loop: form full (q reds + p blues) fairlets
    while (R - r0) - (B - b0) >= q - p and R - r0 >= q and B - b0 >= p:
        cost += _make_fairlet(reds[r0:r0+q] + blues[b0:b0+p],
                              dataset, FAIRLETS, FAIRLET_CENTERS)
        r0 += q
        b0 += p

    # Leftover handling — 3 cases, nothing dropped:
    remaining_total = (R - r0) + (B - b0)

    if remaining_total >= 1 and remaining_total <= p + q:
        # Case 1: everything left fits in one fairlet
        cost += _make_fairlet(reds[r0:] + blues[b0:],
                              dataset, FAIRLETS, FAIRLET_CENTERS)
        r0 = R
        b0 = B

    elif (R - r0) != (B - b0) and (B - b0) >= p:
        # Case 2: unequal remainder — form one mixed fairlet to re-balance
        take_r = (R - r0) - (B - b0) + p
        cost  += _make_fairlet(reds[r0:r0+take_r] + blues[b0:b0+p],
                               dataset, FAIRLETS, FAIRLET_CENTERS)
        r0 += take_r
        b0 += p

    # Case 3: equal remainder — pair 1:1
    assert R - r0 == B - b0, \
        f"Unequal remainder after fairlet formation: reds={R-r0}, blues={B-b0}"
    for i in range(R - r0):
        cost += _make_fairlet([reds[r0 + i], blues[b0 + i]],
                              dataset, FAIRLETS, FAIRLET_CENTERS)
    return cost


print("Fairlet helper functions defined.")

Fairlet helper functions defined.


In [7]:
# ── Twagner Node Fairlet Decomposition (3-phase leftover bubbling) ──

def _node_fairlet_decomposition(p, q, node, dataset, donelist,
                                 FAIRLETS, FAIRLET_CENTERS, depth=0):
    """
    Recursively decompose fairlets along the quadtree.
    Uses 3 phases to bubble excess/unbalanced points UP to the parent
    so that each node receives a balanced set.

    Phase 1 — must-remove: points that make a child too imbalanced
    Phase 2 — may-remove : optional points to help balance the parent
    Phase 3 — unsaturated: excess from incomplete fairlets

    This guarantees no point is ever dropped.
    """

    # ── Leaf node ──────────────────────────────────────────────
    if len(node.children) == 0:
        node.reds  = [i for i in node.reds  if donelist[i] == 0]
        node.blues = [i for i in node.blues if donelist[i] == 0]
        assert _balanced(p, q, len(node.reds), len(node.blues)), \
            f"Unbalanced leaf: reds={len(node.reds)}, blues={len(node.blues)}"
        return _basic_fairlet_decomposition(
            p, q, node.blues, node.reds, dataset, FAIRLETS, FAIRLET_CENTERS
        )

    # ── Filter already-processed points from children ──────────
    for child in node.children:
        child.reds  = [i for i in child.reds  if donelist[i] == 0]
        child.blues = [i for i in child.blues if donelist[i] == 0]

    R = [len(child.reds)  for child in node.children]
    B = [len(child.blues) for child in node.children]

    # Nothing left in this subtree
    if sum(R) == 0 and sum(B) == 0:
        return 0
    if sum(R) == 0 or sum(B) == 0:
        assert sum(R) == 0 and sum(B) == 0, \
            "One colour class became empty while the other did not"
        return 0

    NR = 0   # points bubbled up to this node (red)
    NB = 0   # points bubbled up to this node (blue)

    # ── Phase 1: Must-remove — lift points that over-balance a child ─
    for i in range(len(node.children)):
        if R[i] >= B[i]:
            must_remove_red = max(0, R[i] - int(np.floor(B[i] * q * 1. / p)))
            R[i]  -= must_remove_red
            NR    += must_remove_red
        else:
            must_remove_blue = max(0, B[i] - int(np.floor(R[i] * q * 1. / p)))
            B[i]  -= must_remove_blue
            NB    += must_remove_blue

    # How many more points are needed to balance (NR, NB)?
    if NR >= NB:
        missing = max(0, int(np.ceil(NR * p * 1. / q)) - NB)
    else:
        missing = max(0, int(np.ceil(NB * p * 1. / q)) - NR)

    # ── Phase 2: May-remove — optionally lift more to balance parent ─
    for i in range(len(node.children)):
        if missing == 0:
            assert _balanced(p, q, NR, NB)
            break
        if NR >= NB:
            may_remove_blue = B[i] - int(np.ceil(R[i] * p * 1. / q))
            remove_blue     = min(may_remove_blue, missing)
            B[i]  -= remove_blue
            NB    += remove_blue
            missing -= remove_blue
        else:
            may_remove_red = R[i] - int(np.ceil(B[i] * p * 1. / q))
            remove_red     = min(may_remove_red, missing)
            R[i]  -= remove_red
            NR    += remove_red
            missing -= remove_red

    # ── Phase 3: Unsaturated — lift excess from incomplete fairlets ──
    for i in range(len(node.children)):
        if _balanced(p, q, NR, NB):
            break
        if R[i] >= B[i]:
            num_saturated  = int(R[i] / q)
            excess_red     = R[i] - q * num_saturated
            excess_blue    = B[i] - p * num_saturated
        else:
            num_saturated  = int(B[i] / q)
            excess_red     = R[i] - p * num_saturated
            excess_blue    = B[i] - q * num_saturated
        R[i]  -= excess_red
        NR    += excess_red
        B[i]  -= excess_blue
        NB    += excess_blue

    assert _balanced(p, q, NR, NB), \
        f"Constructed node sets are unbalanced: NR={NR}, NB={NB}"

    # Collect the bubbled-up points and mark as done
    reds  = []
    blues = []
    for i in range(len(node.children)):
        for j in node.children[i].reds[R[i]:]:
            reds.append(j)
            donelist[j] = 1
        for j in node.children[i].blues[B[i]:]:
            blues.append(j)
            donelist[j] = 1

    assert len(reds) == NR and len(blues) == NB

    # Form fairlets from bubbled-up points at this level
    # + recurse into children for their remaining (kept) points
    return (
        _basic_fairlet_decomposition(
            p, q, blues, reds, dataset, FAIRLETS, FAIRLET_CENTERS
        )
        + sum(
            _node_fairlet_decomposition(
                p, q, child, dataset, donelist, FAIRLETS, FAIRLET_CENTERS, depth + 1
            )
            for child in node.children
        )
    )


print("Node fairlet decomposition defined.")

Node fairlet decomposition defined.


In [8]:
# ── Twagner Main Entry Point ──────────────────────────────────

def tree_fairlet_decomposition(p, q, dataset, colors):
    """
    Full Twagner pipeline:
      1. Build a quadtree over the dataset
      2. Decompose fairlets along the tree (distance-aware, no drops)

    Parameters
    ----------
    p, q     : balance parameters (p <= q, gcd=1)
    dataset  : (n, d) dense float64 numpy array
    colors   : list/array of 0/1 group labels (length n)
               NOTE: internally 0=reds, 1=blues (Twagner convention)
               We pass sensitive directly: 1=married=red, 0=others=blue
               so we flip: pass (1 - sensitive) as colors to match convention

    Returns
    -------
    FAIRLETS        : list of fairlets (each a list of data indices)
    FAIRLET_CENTERS : list of medoid indices (one per fairlet)
    cost            : total intra-fairlet distance cost
    """
    assert p <= q, "Please use balance parameters with p <= q"

    FAIRLETS        = []
    FAIRLET_CENTERS = []

    print("Building quadtree...")
    root = build_quadtree(dataset)

    # Populate red/blue lists bottom-up
    root.populate_colors(colors)

    n_reds  = len(root.reds)
    n_blues = len(root.blues)
    print(f"  Tree reds (sensitive=1) : {n_reds}")
    print(f"  Tree blues (sensitive=0): {n_blues}")

    assert _balanced(p, q, n_reds, n_blues), \
        f"Dataset is globally unbalanced: reds={n_reds}, blues={n_blues}. " \
        f"Consider adjusting p/q parameters."

    donelist = [0] * dataset.shape[0]

    print("Running tree fairlet decomposition...")
    cost = _node_fairlet_decomposition(
        p, q, root, dataset, donelist, FAIRLETS, FAIRLET_CENTERS
    )

    # Verify all points were assigned
    assigned = sum(len(f) for f in FAIRLETS)
    print(f"  Fairlets formed         : {len(FAIRLETS)}")
    print(f"  Points assigned         : {assigned} / {dataset.shape[0]}")
    print(f"  Fairlet centers         : {len(FAIRLET_CENTERS)}")
    print(f"  Total intra-fairlet cost: {cost:.4f}")

    return FAIRLETS, FAIRLET_CENTERS, cost


print("tree_fairlet_decomposition defined.")

tree_fairlet_decomposition defined.


---
## Cell 5 — Fairlet Medoid Computation
*(unchanged — Twagner already computes medoids inside `_make_fairlet`,
but we keep this for compatibility with the rest of the pipeline)*

In [9]:
def compute_fairlet_centers(X, fairlets):
    """
    For each fairlet, find the medoid (point minimising total intra-fairlet distance).
    When using Twagner's pipeline, FAIRLET_CENTERS already contains these indices.
    This function provides an explicit recomputation for verification.

    Returns
    -------
    centers : array of original data indices that are fairlet medoids
    """
    centers = []
    for fl in fairlets:
        pts          = X[fl]
        D            = cdist(pts, pts)
        medoid_index = np.argmin(D.sum(axis=1))
        centers.append(fl[medoid_index])
    return np.array(centers)

---
## Cell 6 — K-Medoids (PAM)

In [10]:
def kmedoids(X, k, max_iter=100, random_state=42):
    """
    K-Medoids clustering using the PAM (Partitioning Around Medoids) algorithm.
    Uses Euclidean distance.

    Returns
    -------
    labels         : cluster assignment for each point
    medoid_indices : indices of final medoids
    """
    np.random.seed(random_state)
    n = X.shape[0]

    medoid_indices = np.random.choice(n, k, replace=False)

    for _ in range(max_iter):
        distances = cdist(X, X[medoid_indices])
        labels    = np.argmin(distances, axis=1)

        new_medoids = []
        for i in range(k):
            cluster_points = np.where(labels == i)[0]
            if len(cluster_points) == 0:
                new_medoids.append(medoid_indices[i])
                continue
            cluster_distances = cdist(X[cluster_points], X[cluster_points])
            best_medoid       = cluster_points[np.argmin(cluster_distances.sum(axis=1))]
            new_medoids.append(best_medoid)

        new_medoids = np.array(new_medoids)
        if np.all(new_medoids == medoid_indices):
            break
        medoid_indices = new_medoids

    return labels, medoid_indices

---
## Cell 7 — K-Medians

In [11]:
def kmedians(X, k, max_iter=100, random_state=42):
    """
    K-Medians clustering.
    Uses L1 (Manhattan) distance for assignment; updates centres with the median.

    Returns
    -------
    labels  : cluster assignment for each point
    centers : final median centres (not actual data points)
    """
    np.random.seed(random_state)
    n_samples = X.shape[0]

    indices = np.random.choice(n_samples, k, replace=False)
    centers = X[indices]

    for _ in range(max_iter):
        distances = cdist(X, centers, metric='cityblock')
        labels    = np.argmin(distances, axis=1)

        new_centers = []
        for i in range(k):
            cluster_points = X[labels == i]
            if len(cluster_points) == 0:
                new_centers.append(centers[i])
            else:
                new_centers.append(np.median(cluster_points, axis=0))

        new_centers = np.array(new_centers)
        if np.allclose(new_centers, centers):
            break
        centers = new_centers

    return labels, centers

---
## Cell 8 — Label Assignment from Fairlets

In [12]:
def assign_labels_from_fairlets(fairlets, center_labels, n):
    """
    Propagate the cluster label of each fairlet's medoid back to all
    points in that fairlet.

    Parameters
    ----------
    fairlets      : list of index lists
    center_labels : cluster label assigned to each fairlet centre
    n             : total number of data points

    Returns
    -------
    labels : full-length cluster assignment array
    """
    labels = np.zeros(n, dtype=int)
    for fl, lab in zip(fairlets, center_labels):
        for idx in fl:
            labels[idx] = lab
    return labels

---
## Cell 9 — Fairness Metrics

In [13]:
def fairness_metrics(labels, sensitive):
    """
    Compute fairness metrics for a clustering result.

    Metrics
    -------
    min_balance    : min ratio of minority/majority per cluster
                     (strict Chierichetti et al. definition — worst cluster)
    avg_balance    : mean ratio across clusters (softer summary)
    violation_rate : fraction of clusters violating the p/q balance target
    avg_dp_gap     : mean |cluster_ratio - global_ratio| (demographic parity)

    Returns
    -------
    dict of metric name -> value
    """
    K            = len(np.unique(labels))
    balances     = []
    violations   = 0
    global_ratio = sensitive.mean()
    dp_gaps      = []

    for k in range(K):
        mask  = labels == k
        group = sensitive[mask]

        n0 = np.sum(group == 0)
        n1 = np.sum(group == 1)

        if max(n0, n1) == 0:
            continue

        balance = min(n0, n1) / max(n0, n1)
        balances.append(balance)

        if balance < (P / Q):
            violations += 1

        cluster_ratio = group.mean()
        dp_gaps.append(abs(cluster_ratio - global_ratio))

    return {
        "min_balance"    : np.min(balances),    # strict Chierichetti definition
        "avg_balance"    : np.mean(balances),   # softer summary
        "violation_rate" : violations / K,
        "avg_dp_gap"     : np.mean(dp_gaps)
    }

---
## Cell 10 — Explainability Metrics (Decision Tree)

In [14]:
def explainability_metrics(X, labels, feature_names, print_rules=True):
    """
    Fit a shallow decision tree to approximate the clustering and
    report rule-based explainability metrics.

    Metrics
    -------
    tree_fidelity : accuracy of the decision tree reproducing cluster labels
    tree_depth    : depth of the fitted tree
    tree_leaves   : number of leaf nodes

    Returns
    -------
    dict of metric name -> value
    """
    clf = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
    clf.fit(X, labels)

    if print_rules:
        rules = export_text(clf, feature_names=feature_names)
        print("\n--- DECISION TREE RULES ---")
        print(rules)

    return {
        "tree_fidelity" : clf.score(X, labels),
        "tree_depth"    : clf.get_depth(),
        "tree_leaves"   : clf.get_n_leaves()
    }

---
## Cell 11 — Unified Evaluation Function

In [ ]:
def evaluate_model(name, X, labels, sensitive, feature_names,
                   print_rules=False):
    """
    Print clustering quality, fairness, and explainability metrics.

    Returns
    -------
    results dict with all metric values
    """
    print(f"\n{'='*55}")
    print(f"  RESULTS: {name}")
    print(f"{'='*55}")

    sil = silhouette_score(X, labels, sample_size=5000, random_state=RANDOM_STATE)
    db  = davies_bouldin_score(X, labels)
    print(f"\n[CLUSTERING QUALITY]")
    print(f"  Silhouette Score   : {sil:.4f}  (higher = better)")
    print(f"  Davies-Bouldin Idx : {db:.4f}  (lower  = better)")

    fm = fairness_metrics(labels, sensitive)
    print(f"\n[FAIRNESS]")
    print(f"  min_balance        : {fm['min_balance']:.4f}  "
          f"← strict Chierichetti (worst cluster)")
    print(f"  avg_balance        : {fm['avg_balance']:.4f}  "
          f"← softer summary across all clusters")
    print(f"  violation_rate     : {fm['violation_rate']:.4f}")
    print(f"  avg_dp_gap         : {fm['avg_dp_gap']:.4f}")

    em = explainability_metrics(X, labels, feature_names,
                                print_rules=print_rules)
    print(f"\n[EXPLAINABILITY]")
    print(f"  Tree Fidelity      : {em['tree_fidelity']:.4f}")
    print(f"  Tree Depth         : {em['tree_depth']}")
    print(f"  Tree Leaves        : {em['tree_leaves']}")

    return {
        "model"          : name,
        "silhouette"     : sil,
        "davies_bouldin" : db,
        **fm,
        **em
    }


---
## Cell 12 — Baseline: K-Means → Metrics

In [16]:
# ── Baseline 1: K-Means ──────────────────────────────────────
kmeans = KMeans(
    n_clusters=K_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=10
)
kmeans_labels = kmeans.fit_predict(X)

results_kmeans = evaluate_model(
    "BASELINE: K-Means",
    X, kmeans_labels, sensitive, feature_names
)


  RESULTS: BASELINE: K-Means


MemoryError: Unable to allocate 1.00 GiB for an array with shape (4473, 30000) and data type float64

---
## Cell 13 — Baseline: K-Medians → Metrics

In [ ]:
# ── Baseline 2: K-Medians ────────────────────────────────────
kmedians_labels, kmedians_centers = kmedians(
    X,
    K_CLUSTERS,
    random_state=RANDOM_STATE
)

results_kmedians = evaluate_model(
    "BASELINE: K-Medians",
    X, kmedians_labels, sensitive, feature_names
)


  RESULTS: BASELINE: K-Medians

[CLUSTERING QUALITY]
  Silhouette Score   : 0.0761  (higher = better)
  Davies-Bouldin Idx : 3.2772  (lower  = better)

[FAIRNESS]
  min_balance        : 0.0510  ← strict Chierichetti (worst cluster)
  avg_balance        : 0.3606  ← softer summary across all clusters
  violation_rate     : 0.8000
  avg_dp_gap         : 0.2675

[EXPLAINABILITY (Rule-Based)]
  tree_fidelity       : 0.8492
  tree_depth          : 4
  tree_leaves         : 15


---
## Cell 14 — Twagner Fairlets + K-Medians → Metrics

**Key differences from old Cell 13:**
- `tree_fairlet_decomposition` builds a quadtree first
- Fairlets are formed among geometrically close points
- Leftover points bubble up the tree — nothing is dropped
- `colors` uses Twagner's convention (0=reds/sensitive=1, 1=blues/sensitive=0)
  so we pass `sensitive` directly (1=married → internally treated as red)

In [ ]:
# ── Proposed: Twagner Fairlets + K-Medians ───────────────────

# Step 1: Run Twagner tree fairlet decomposition
# colors = sensitive directly:
#   sensitive=1 (married) → Twagner 'reds'
#   sensitive=0 (others)  → Twagner 'blues'
FAIRLETS, FAIRLET_CENTERS, fairlet_cost = tree_fairlet_decomposition(
    p=P,
    q=Q,
    dataset=X,
    colors=sensitive.tolist()
)

# Step 2: FAIRLET_CENTERS already contains medoid indices (from Twagner)
# We convert to numpy array for indexing
fairlet_center_indices = np.array(FAIRLET_CENTERS)

print(f"\nVerification — fairlet center indices sample: {fairlet_center_indices[:10]}")

# Step 3: Cluster the fairlet medoids with K-Medians
center_labels, _ = kmedians(
    X[fairlet_center_indices],
    K_CLUSTERS,
    random_state=RANDOM_STATE
)

# Step 4: Propagate labels back to all points via fairlet membership
fair_labels = assign_labels_from_fairlets(
    FAIRLETS,
    center_labels,
    n=len(X)
)

print(f"\nLabel distribution: "
      f"{dict(zip(*np.unique(fair_labels, return_counts=True)))}")

results_fair = evaluate_model(
    "TWAGNER FAIRLETS + K-Medians",
    X, fair_labels, sensitive, feature_names
)

Building quadtree...
  Tree reds (sensitive=1) : 13317
  Tree blues (sensitive=0): 16683
Running tree fairlet decomposition...
  Fairlets formed         : 13317
  Points assigned         : 30000 / 30000
  Fairlet centers         : 13317
  Total intra-fairlet cost: 64343.6893

Verification — fairlet center indices sample: [    1   232  1790 15741  5975  1095  1160  1325  3607 14916]

Label distribution: {np.int64(0): np.int64(3333), np.int64(1): np.int64(7410), np.int64(2): np.int64(4846), np.int64(3): np.int64(8781), np.int64(4): np.int64(5630)}

  RESULTS: TWAGNER FAIRLETS + K-Medians

[CLUSTERING QUALITY]
  Silhouette Score   : 0.0032  (higher = better)
  Davies-Bouldin Idx : 6.2328  (lower  = better)

[FAIRNESS]
  min_balance        : 0.6942  ← strict Chierichetti (worst cluster)
  avg_balance        : 0.8289  ← softer summary across all clusters
  violation_rate     : 0.0000
  avg_dp_gap         : 0.0248

[EXPLAINABILITY (Rule-Based)]
  tree_fidelity       : 0.5784333333333334
  tr

---
## Cell 15 — Explainability Analysis (Decision Rules printed)

In [ ]:
# ── Decision Tree Rules: K-Means ─────────────────────────────
print("=" * 55)
print("  DECISION TREE RULES: K-Means")
print("=" * 55)
_ = explainability_metrics(X, kmeans_labels, feature_names, print_rules=True)

  DECISION TREE RULES: K-Means

--- DECISION TREE RULES ---
|--- num__day <= 0.08
|   |--- num__pdays <= 0.45
|   |   |--- num__duration <= 1.37
|   |   |   |--- num__campaign <= 2.17
|   |   |   |   |--- class: 0
|   |   |   |--- num__campaign >  2.17
|   |   |   |   |--- class: 3
|   |   |--- num__duration >  1.37
|   |   |   |--- num__duration <= 1.56
|   |   |   |   |--- class: 2
|   |   |   |--- num__duration >  1.56
|   |   |   |   |--- class: 2
|   |--- num__pdays >  0.45
|   |   |--- num__pdays <= 0.86
|   |   |   |--- num__previous <= 0.76
|   |   |   |   |--- class: 0
|   |   |   |--- num__previous >  0.76
|   |   |   |   |--- class: 4
|   |   |--- num__pdays >  0.86
|   |   |   |--- num__duration <= 2.89
|   |   |   |   |--- class: 4
|   |   |   |--- num__duration >  2.89
|   |   |   |   |--- class: 2
|--- num__day >  0.08
|   |--- num__pdays <= 0.69
|   |   |--- num__campaign <= 1.85
|   |   |   |--- num__duration <= 1.26
|   |   |   |   |--- class: 1
|   |   |   |--- num__

In [ ]:
# ── Decision Tree Rules: K-Medians ───────────────────────────
print("=" * 55)
print("  DECISION TREE RULES: K-Medians")
print("=" * 55)
_ = explainability_metrics(X, kmedians_labels, feature_names, print_rules=True)

  DECISION TREE RULES: K-Medians

--- DECISION TREE RULES ---
|--- cat__contact_unknown <= 0.50
|   |--- cat__education_tertiary <= 0.50
|   |   |--- num__pdays <= 0.69
|   |   |   |--- cat__loan_no <= 0.50
|   |   |   |   |--- class: 4
|   |   |   |--- cat__loan_no >  0.50
|   |   |   |   |--- class: 2
|   |   |--- num__pdays >  0.69
|   |   |   |--- cat__housing_no <= 0.50
|   |   |   |   |--- class: 1
|   |   |   |--- cat__housing_no >  0.50
|   |   |   |   |--- class: 1
|   |--- cat__education_tertiary >  0.50
|   |   |--- num__pdays <= 0.69
|   |   |   |--- cat__loan_yes <= 0.50
|   |   |   |   |--- class: 3
|   |   |   |--- cat__loan_yes >  0.50
|   |   |   |   |--- class: 4
|   |   |--- num__pdays >  0.69
|   |   |   |--- cat__housing_yes <= 0.50
|   |   |   |   |--- class: 3
|   |   |   |--- cat__housing_yes >  0.50
|   |   |   |   |--- class: 1
|--- cat__contact_unknown >  0.50
|   |--- cat__housing_no <= 0.50
|   |   |--- num__pdays <= 0.87
|   |   |   |--- cat__poutcome_fail

In [ ]:
# ── Decision Tree Rules: Twagner Fairlets + K-Medians ────────
print("=" * 55)
print("  DECISION TREE RULES: Twagner Fairlets + K-Medians")
print("=" * 55)
_ = explainability_metrics(X, fair_labels, feature_names, print_rules=True)

  DECISION TREE RULES: Twagner Fairlets + K-Medians

--- DECISION TREE RULES ---
|--- num__pdays <= 0.69
|   |--- cat__contact_unknown <= 0.50
|   |   |--- cat__education_secondary <= 0.50
|   |   |   |--- cat__housing_no <= 0.50
|   |   |   |   |--- class: 2
|   |   |   |--- cat__housing_no >  0.50
|   |   |   |   |--- class: 3
|   |   |--- cat__education_secondary >  0.50
|   |   |   |--- cat__housing_no <= 0.50
|   |   |   |   |--- class: 1
|   |   |   |--- cat__housing_no >  0.50
|   |   |   |   |--- class: 3
|   |--- cat__contact_unknown >  0.50
|   |   |--- cat__month_may <= 0.50
|   |   |   |--- cat__loan_yes <= 0.50
|   |   |   |   |--- class: 1
|   |   |   |--- cat__loan_yes >  0.50
|   |   |   |   |--- class: 0
|   |   |--- cat__month_may >  0.50
|   |   |   |--- cat__housing_no <= 0.50
|   |   |   |   |--- class: 3
|   |   |   |--- cat__housing_no >  0.50
|   |   |   |   |--- class: 2
|--- num__pdays >  0.69
|   |--- cat__housing_no <= 0.50
|   |   |--- num__pdays <= 1.48
| 

---
## Cell 16 — Summary Comparison Table

Key columns to compare:
- `min_balance` : strict Chierichetti definition — **Twagner fairlets should improve this over baselines**
- `avg_balance` : softer summary
- `silhouette`  : clustering quality (fairness may trade this off slightly)
- `tree_fidelity`: how well rules explain the clustering

In [ ]:
# ── Side-by-side comparison ───────────────────────────────────
summary = pd.DataFrame([
    results_kmeans,
    results_kmedians,
    results_fair
]).set_index('model')

summary = summary.round(4)

print("\n=== FULL COMPARISON ===")
print(summary.T.to_string())
print()
summary


=== FULL COMPARISON ===
model           BASELINE: K-Means  BASELINE: K-Medians  TWAGNER FAIRLETS + K-Medians
silhouette                 0.1074               0.0761                        0.0032
davies_bouldin             2.0899               3.2772                        6.2328
min_balance                0.4874               0.0510                        0.6942
avg_balance                0.8035               0.3606                        0.8289
violation_rate             0.2000               0.8000                        0.0000
avg_dp_gap                 0.0428               0.2675                        0.0248
tree_fidelity              0.9564               0.8492                        0.5784
tree_depth                 4.0000               4.0000                        4.0000
tree_leaves               16.0000              15.0000                       16.0000



,silhouette,davies_bouldin,min_balance,avg_balance,violation_rate,avg_dp_gap,tree_fidelity,tree_depth,tree_leaves
model,,,,,,,,,
BASELINE: K-Means,0.1074,2.0899,0.4874,0.8035,0.2,0.0428,0.9564,4,16
BASELINE: K-Medians,0.0761,3.2772,0.0510,0.3606,0.8,0.2675,0.8492,4,15
TWAGNER FAIRLETS + K-Medians,0.0032,6.2328,0.6942,0.8289,0.0,0.0248,0.5784,4,16
